# Generación y carga de datos

En este notebook se generan los datos ficticios necesarios para poblar el modelo de KTE E-commerce y posteriormente se cargan en las tablas correspondientes de BigQuery.

La generación se realiza de forma progresiva, comenzando por las entidades independientes y utilizando los datos previamente creados para mantener las relaciones entre tablas. Para ello se emplean principalmente `Faker` y valores aleatorios controlados, utilizando textos predefinidos en aquellos campos donde `Faker` no ofrece suficiente coherencia semántica.

En el código también se implementa la función para cargar cada tabla de datos desde el DataFrame, asís e pueden idetificar errores en el proces y/o validar la carga correcta.

Una vez generado cada conjunto de datos, se transforma en un DataFrame de Pandas y se carga en su tabla correspondiente de BigQuery.

In [1]:
import os
import random
from datetime import timedelta

import pandas as pd
from faker import Faker
from dotenv import load_dotenv
from google.cloud import bigquery

load_dotenv()

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")

client = bigquery.Client(project=PROJECT_ID)
fake = Faker("es_ES")

def load_dataframe(df, table_name):
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"

    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE"
    )

    try:
        job = client.load_table_from_dataframe(
            df,
            table_id,
            job_config=job_config
        )

        job.result()

        table = client.get_table(table_id)

        print(
            f"Tabla {table_name} cargada correctamente: "
            f"{table.num_rows} registros"
        )

    except Exception as error:
        print(f"Error al cargar la tabla {table_name}: {error}")

print(f"Conectado al proyecto: {client.project}")
print(f"Dataset: {DATASET_ID}")

Conectado al proyecto: project-sql-big-query-epm
Dataset: kte_ecom


## Generación de clientes

In [2]:
NUM_CUSTOMERS = 500

acquisition_channels = [
    "organic",
    "paid_ads",
    "social_media",
    "referral",
    "email",
]

customers = []

for customer_id in range(1, NUM_CUSTOMERS + 1):
    customers.append({
        "customer_id": customer_id,
        "first_name": fake.first_name(),
        "last_name": fake.last_name(),
        "email": fake.unique.email(),
        "phone": fake.phone_number(),
        "country": fake.country(),
        "city": fake.city(),
        "acquisition_channel": random.choice(acquisition_channels),
        "registration_date": fake.date_between(
            start_date="-2y",
            end_date="today"
        ),
    })

df_customers = pd.DataFrame(customers)

df_customers.head()

,customer_id,first_name,last_name,email,phone,country,city,acquisition_channel,registration_date
0,1,Elena,Aguiló,estermanjon@example.com,+34849182395,Líbano,Ávila,social_media,2024-09-18
1,2,Fermín,Miralles,delgadoazeneth@example.net,+34925 66 15 49,Argelia,León,paid_ads,2024-12-17
2,3,María Cristina,Ochoa,rocioquero@example.com,+34922 42 56 09,Timor-Leste,Huelva,paid_ads,2025-08-24
3,4,Evita,Osuna,castejoneugenio@example.net,+34 974 641 550,Etiopía,Baleares,email,2025-08-01
4,5,Violeta,Chaves,ruperta36@example.org,+34 721682492,Nicaragua,Ceuta,social_media,2024-11-19


### Carga de clientes en BigQuery

In [3]:
load_dataframe(df_customers, "customers")

c:\Users\NitroPC\Pictures\KTE\TheBridge IA Engeneering\Docs\VSC\GitHubEPM\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Tabla customers cargada correctamente: 500 registros


## Generación de categorías

In [4]:
categories = [
    {
        "category_id": 1,
        "name": "Smartphones",
        "description": "Teléfonos móviles y smartphones",
    },
    {
        "category_id": 2,
        "name": "Laptops",
        "description": "Ordenadores portátiles",
    },
    {
        "category_id": 3,
        "name": "Audio",
        "description": "Auriculares, altavoces y dispositivos de audio",
    },
    {
        "category_id": 4,
        "name": "Tablets",
        "description": "Tablets y dispositivos similares",
    },
    {
        "category_id": 5,
        "name": "Accesorios",
        "description": "Accesorios y periféricos electrónicos",
    },
]

df_categories = pd.DataFrame(categories)

df_categories

,category_id,name,description
0,1,Smartphones,Teléfonos móviles y smartphones
1,2,Laptops,Ordenadores portátiles
2,3,Audio,"Auriculares, altavoces y dispositivos de audio"
3,4,Tablets,Tablets y dispositivos similares
4,5,Accesorios,Accesorios y periféricos electrónicos


### Carga de categorías en BigQuery

In [5]:
load_dataframe(df_categories, "categories")

c:\Users\NitroPC\Pictures\KTE\TheBridge IA Engeneering\Docs\VSC\GitHubEPM\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Tabla categories cargada correctamente: 5 registros


## Generación de productos

Para los campos de texto descriptivo se utilizan plantillas predefinidas en lugar de `Faker`. Como se ha especificado al inicio del ipynb la generación de frases aleatorias no garantiza coherencia semántica ni descripciones realistas de los productos.

In [19]:
NUM_PRODUCTS = 100

product_descriptions = {
    1: [
        "Smartphone con pantalla de alta resolución y gran autonomía.",
        "Teléfono móvil con cámara avanzada y almacenamiento ampliable.",
        "Smartphone equilibrado para uso diario y multimedia.",
    ],
    2: [
        "Portátil ligero y potente para trabajo y estudio.",
        "Ordenador portátil con buen rendimiento y batería de larga duración.",
        "Laptop versátil para productividad, navegación y entretenimiento.",
    ],
    3: [
        "Dispositivo de audio con sonido claro y buena calidad.",
        "Equipo de audio diseñado para ofrecer una experiencia envolvente.",
        "Accesorio de audio cómodo y adecuado para uso diario.",
    ],
    4: [
        "Tablet ligera con pantalla de alta resolución.",
        "Dispositivo portátil ideal para navegación y contenido multimedia.",
        "Tablet versátil para entretenimiento, estudio y productividad.",
    ],
    5: [
        "Accesorio electrónico práctico y fácil de utilizar.",
        "Periférico diseñado para complementar otros dispositivos.",
        "Accesorio funcional para mejorar la experiencia de uso.",
    ],
}

product_names = {
    1: ["Nova", "Vertex", "Pulse", "Nexus", "Orion"],
    2: ["VertexBook", "NovaBook", "CoreBook", "NexusBook", "OrbitBook"],
    3: ["SonicBeat", "WaveSound", "PulseAudio", "EchoSound", "NovaSound"],
    4: ["NovaTab", "VertexTab", "NexusTab", "OrbitTab", "PulseTab"],
    5: ["PowerLink", "ConnectPro", "NovaGear", "CoreLink", "NexusGear"],
}

products = []

for product_id in range(1, NUM_PRODUCTS + 1):
    category_id = random.randint(1, 5)

    cost_price = round(random.uniform(20, 800), 2)
    sale_price = round(cost_price * random.uniform(1.15, 1.60), 2)

    products.append({
        "product_id": product_id,
        "category_id": category_id,
        "name": f"{random.choice(product_names[category_id])} {product_id}",
        "description": random.choice(
            product_descriptions[category_id]
        ),
        "sale_price": sale_price,
        "cost_price": cost_price,
        "stock": random.randint(0, 200),
        "is_active": random.choice([True, True, True, False]),
    })

df_products = pd.DataFrame(products)

df_products.head()

,product_id,category_id,name,description,sale_price,cost_price,stock,is_active
0,1,2,CoreBook 1,"Laptop versátil para productividad, navegación...",81.65,62.07,112,False
1,2,2,NexusBook 2,Portátil ligero y potente para trabajo y estudio.,1002.64,784.03,62,True
2,3,5,NexusGear 3,Accesorio electrónico práctico y fácil de util...,751.12,525.41,56,True
3,4,5,PowerLink 4,Accesorio funcional para mejorar la experienci...,708.94,525.85,10,True
4,5,5,CoreLink 5,Accesorio funcional para mejorar la experienci...,548.72,468.18,77,False


### Carga de productos en BigQuery

In [20]:
load_dataframe(df_products, "products")

c:\Users\NitroPC\Pictures\KTE\TheBridge IA Engeneering\Docs\VSC\GitHubEPM\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Tabla products cargada correctamente: 100 registros


## Generación de pedidos

In [8]:
NUM_ORDERS = 2000

order_statuses = [
    "pending",
    "confirmed",
    "shipped",
    "delivered",
    "cancelled",
    "returned",
]

today = pd.Timestamp.today().date()

orders = []

for order_id in range(1, NUM_ORDERS + 1):
    customer_id = random.randint(1, NUM_CUSTOMERS)

    customer = df_customers.loc[
        df_customers["customer_id"] == customer_id
    ].iloc[0]

    registration_date = customer["registration_date"]

    order_date = fake.date_between(
        start_date=registration_date,
        end_date="today"
    )

    status = random.choice(order_statuses)

    shipping_date = None
    delivery_date = None

    if status in ["shipped", "delivered", "returned"]:
        candidate_shipping = order_date + timedelta(
            days=random.randint(1, 3)
        )

        if candidate_shipping <= today:
            shipping_date = candidate_shipping
        else:
            status = random.choice(["pending", "confirmed"])

    if status in ["delivered", "returned"]:
        candidate_delivery = shipping_date + timedelta(
            days=random.randint(1, 5)
        )

        if candidate_delivery <= today:
            delivery_date = candidate_delivery
        else:
            status = "shipped"

    orders.append({
        "order_id": order_id,
        "customer_id": customer_id,
        "status": status,
        "shipping_address": fake.street_address(),
        "shipping_city": fake.city(),
        "shipping_country": fake.country(),
        "order_date": order_date,
        "shipping_date": shipping_date,
        "delivery_date": delivery_date,
    })

df_orders = pd.DataFrame(orders)

df_orders.head()

,order_id,customer_id,status,shipping_address,shipping_city,shipping_country,order_date,shipping_date,delivery_date
0,1,498,pending,Glorieta de Eli Contreras 49 Puerta 5,Almería,República Dominicana,2026-02-03,None,None
1,2,220,returned,Acceso de Rosenda Garay 51,Santa Cruz de Tenerife,Eslovaquia,2026-08-19,2026-08-20,2026-08-25
2,3,422,delivered,Rambla de Fabricio Gámez 599,Albacete,Vanuatu,2026-07-07,2026-07-09,2026-07-14
3,4,455,confirmed,Cuesta de Yaiza Martí 22 Piso 4,Badajoz,Santo Tomé y Príncipe,2026-01-23,None,None
4,5,183,returned,Alameda Corona Esteve 46,La Coruña,República Centroafricana,2026-08-16,2026-08-18,2026-08-22


### Carga de pedidos en BigQuery

Como estamos generando un dataset sintético completo, cada ejecución debe sustituir los datos anteriores. Se usa WRITE_TRUNCATE para que el notebook sea reproducible y no duplique registros si se ejecuta más de una vez.

In [9]:
load_dataframe(df_orders, "orders")

c:\Users\NitroPC\Pictures\KTE\TheBridge IA Engeneering\Docs\VSC\GitHubEPM\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Tabla orders cargada correctamente: 2000 registros


## Generación de líneas de pedido

In [10]:
order_items = []
order_item_id = 1

for order_id in range(1, NUM_ORDERS + 1):
    num_items = random.choice([1, 2, 2, 2, 2, 3, 3, 3])
    selected_products = random.sample(range(1, NUM_PRODUCTS + 1), num_items)

    for product_id in selected_products:
        product = df_products.loc[
            df_products["product_id"] == product_id
        ].iloc[0]

        order_items.append({
            "order_item_id": order_item_id,
            "order_id": order_id,
            "product_id": product_id,
            "quantity": random.randint(1, 3),
            "unit_price": product["sale_price"],
            "discount": random.choice([0, 0, 0, 0.05, 0.10, 0.15]),
        })

        order_item_id += 1

df_order_items = pd.DataFrame(order_items)

df_order_items.head()

,order_item_id,order_id,product_id,quantity,unit_price,discount
0,1,1,92,3,839.21,0.10
1,2,2,25,1,847.67,0.05
2,3,2,8,3,342.49,0.05
3,4,2,88,2,545.62,0.00
4,5,3,82,3,635.36,0.00


### Carga de líneas de pedido en BigQuery

In [11]:
load_dataframe(df_order_items, "order_items")

c:\Users\NitroPC\Pictures\KTE\TheBridge IA Engeneering\Docs\VSC\GitHubEPM\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Tabla order_items cargada correctamente: 4510 registros


## Generación de pagos

In [12]:
payment_methods = [
    "card",
    "paypal",
    "bank_transfer",
]

payments = []

for payment_id in range(1, NUM_ORDERS + 1):
    order_id = payment_id

    order_lines = df_order_items[
        df_order_items["order_id"] == order_id
    ]

    amount = (
        order_lines["quantity"]
        * order_lines["unit_price"]
        * (1 - order_lines["discount"])
    ).sum()

    order = df_orders.loc[
        df_orders["order_id"] == order_id
    ].iloc[0]

    if order["status"] == "pending":
        payment_status = "pending"
    elif order["status"] == "cancelled":
        payment_status = random.choice(["failed", "refunded"])
    elif order["status"] == "returned":
        payment_status = "refunded"
    else:
        payment_status = "completed"

    payments.append({
        "payment_id": payment_id,
        "order_id": order_id,
        "payment_method": random.choice(payment_methods),
        "status": payment_status,
        "amount": round(amount, 2),
        "payment_date": order["order_date"],
    })

df_payments = pd.DataFrame(payments)

df_payments.head()

,payment_id,order_id,payment_method,status,amount,payment_date
0,1,1,paypal,pending,2265.87,2026-02-03
1,2,2,paypal,refunded,2872.62,2026-08-19
2,3,3,paypal,completed,5288.82,2026-07-07
3,4,4,paypal,completed,2856.93,2026-01-23
4,5,5,paypal,refunded,765.28,2026-08-16


### Carga de pagos en BigQuery

In [13]:
load_dataframe(df_payments, "payments")

c:\Users\NitroPC\Pictures\KTE\TheBridge IA Engeneering\Docs\VSC\GitHubEPM\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Tabla payments cargada correctamente: 2000 registros


## Generación de valoraciones

Para los comentarios de las valoraciones se utilizan textos predefinidos asociados a cada puntuación como se ha comentado al inicio del ipynb, ya que `Faker` no garantiza que el contenido sea coherente con la valoración otorgada.

In [14]:
review_comments = {
    1: [
        "El producto no cumple con lo esperado.",
        "Muy mala calidad, no lo recomiendo.",
        "El producto llegó con problemas.",
    ],
    2: [
        "La calidad podría ser mejor.",
        "No estoy del todo satisfecho con la compra.",
        "Esperaba más por este precio.",
    ],
    3: [
        "Producto correcto, cumple su función.",
        "Está bien, aunque tiene aspectos mejorables.",
        "Una compra aceptable en general.",
    ],
    4: [
        "Buen producto y buena relación calidad-precio.",
        "Estoy satisfecho con la compra.",
        "Funciona muy bien y cumple lo esperado.",
    ],
    5: [
        "Excelente producto, totalmente recomendado.",
        "Muy satisfecho con la compra.",
        "Gran calidad y funcionamiento perfecto.",
    ],
}

reviews = []
review_id = 1

delivered_orders = df_orders[
    df_orders["status"].isin(["delivered", "returned"])
]["order_id"]

reviewable_items = df_order_items[
    df_order_items["order_id"].isin(delivered_orders)
]

for _, item in reviewable_items.iterrows():
    if random.random() < 0.35:
        rating = random.randint(1, 5)

        order = df_orders.loc[
            df_orders["order_id"] == item["order_id"]
        ].iloc[0]

        review_date = fake.date_between(
            start_date=order["delivery_date"],
            end_date="today"
        )

        reviews.append({
            "review_id": review_id,
            "order_item_id": int(item["order_item_id"]),
            "rating": rating,
            "comment": (
                random.choice(review_comments[rating])
                if random.random() < 0.8
                else None
            ),
            "review_date": review_date,
        })

        review_id += 1

df_reviews = pd.DataFrame(reviews)

df_reviews.head()

,review_id,order_item_id,rating,comment,review_date
0,1,2,3,NaN,2026-08-28
1,2,3,3,"Está bien, aunque tiene aspectos mejorables.",2026-08-27
2,3,10,1,El producto no cumple con lo esperado.,2026-08-25
3,4,17,1,El producto llegó con problemas.,2026-08-23
4,5,33,5,"Excelente producto, totalmente recomendado.",2026-07-28


### Carga de valoraciones en BigQuery

In [15]:
load_dataframe(df_reviews, "reviews")

c:\Users\NitroPC\Pictures\KTE\TheBridge IA Engeneering\Docs\VSC\GitHubEPM\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Tabla reviews cargada correctamente: 480 registros


### Validación de coherencia temporal

Se comprueba que las fechas generadas respeten una secuencia temporal lógica: los pedidos no pueden realizarse antes del registro del cliente y las fechas de envío o entrega no deben situarse en el futuro.

Estas validaciones permiten detectar incoherencias en los datos sintéticos antes de cargarlos definitivamente en BigQuery.

In [16]:
print("Pedidos anteriores al registro del cliente:")

check_dates = df_orders.merge(
    df_customers[["customer_id", "registration_date"]],
    on="customer_id"
)

print((check_dates["order_date"] < check_dates["registration_date"]).sum())


from datetime import date

today = date.today()

print("Envíos en el futuro:")
print(
    (df_orders["shipping_date"].notna() & (df_orders["shipping_date"] > today)).sum()
)

print("Entregas en el futuro:")
print(
    (df_orders["delivery_date"].notna() & (df_orders["delivery_date"] > today)).sum()
)

Pedidos anteriores al registro del cliente:
0
Envíos en el futuro:
0
Entregas en el futuro:
0


In [17]:
print("Valoraciones anteriores a la entrega:")

check_reviews = (
    df_reviews
    .merge(
        df_order_items[["order_item_id", "order_id"]],
        on="order_item_id"
    )
    .merge(
        df_orders[["order_id", "delivery_date"]],
        on="order_id"
    )
)

print(
    (check_reviews["review_date"] < check_reviews["delivery_date"]).sum()
)

Valoraciones anteriores a la entrega:
0


Se comprueba el número de registros generados para cada entidad para verificar que se cumplen los volúmenes mínimos requeridos.

In [18]:
print("Clientes:", len(df_customers))
print("Categorías:", len(df_categories))
print("Productos:", len(df_products))
print("Pedidos:", len(df_orders))
print("Líneas de pedido:", len(df_order_items))
print("Pagos:", len(df_payments))
print("Valoraciones:", len(df_reviews))

Clientes: 500
Categorías: 5
Productos: 100
Pedidos: 2000
Líneas de pedido: 4510
Pagos: 2000
Valoraciones: 480
